## Data Extraction

In [44]:
import pandas as pd

df = pd.read_csv('european_housing_prices_clean.csv')
#print(df.head())

country_list = df['country'].unique()
#print(country_list)

#Remove countries with Euro area or European Union in the name
countries_to_remove = [country for country in country_list if 'Euro area' in country or 'European Union' in country or 'Türkiye' in country]
#print(countries_to_remove)
df = df[~df['country'].isin(countries_to_remove)]
country_list = df['country'].unique()
#print(country_list)

#Change Czechia to Czech Republic
df['country'] = df['country'].replace('Czechia', 'Czech Republic')
country_list = df['country'].unique()
print(country_list)

df['dt'] = df['year'].astype(str) + '-0' + df['quarter_num'].astype(str) + '-01'
df['dt'] = pd.to_datetime(df['dt'], format='%Y-%m-%d')

['Austria' 'Belgium' 'Bulgaria' 'Croatia' 'Cyprus' 'Czech Republic'
 'Denmark' 'Estonia' 'Finland' 'France' 'Germany' 'Hungary' 'Iceland'
 'Ireland' 'Italy' 'Latvia' 'Lithuania' 'Luxembourg' 'Malta' 'Netherlands'
 'Norway' 'Poland' 'Portugal' 'Romania' 'Slovakia' 'Slovenia' 'Spain'
 'Sweden' 'Switzerland']


In [46]:
files = {
    'df_snk': 'european_net_income_single_no_kids.xlsx',
    'df_dnk': 'european_net_double_income_no_kids.xlsx',
    'df_dwk': 'european_net_double_income_with_kids.xlsx'
}

dataframes = {}

for key, file_name in files.items():
    df_inc = pd.read_excel(file_name)
    dataframes[key] = df_inc[df_inc['Name'].isin(country_list)]

# Now you can access them like this:
df_snk = dataframes['df_snk']
df_dnk = dataframes['df_dnk']
df_dwk = dataframes['df_dwk']

country_list_snk = df_snk['Name'].unique()
print(country_list_snk)
print(df_snk.columns)


years = ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

for name, df_1 in dataframes.items():
    # 1. Convert year columns to numeric, turning errors into NaN
    df_1[years] = df_1[years].apply(pd.to_numeric, errors='coerce')
    
    # 2. Perform the indexing math
    # We use .div() with axis=0 to divide each row by its 2015 value
    df_1[years] = df_1[years].div(df_1['2015'], axis=0) * 100

# Access your indexed dataframes
df_snk_indexed = dataframes['df_snk']

['Belgium' 'Bulgaria' 'Czech Republic' 'Denmark' 'Germany' 'Estonia'
 'Ireland' 'Spain' 'France' 'Croatia' 'Italy' 'Cyprus' 'Latvia'
 'Lithuania' 'Luxembourg' 'Hungary' 'Malta' 'Netherlands' 'Austria'
 'Poland' 'Portugal' 'Romania' 'Slovenia' 'Slovakia' 'Finland' 'Sweden'
 'Iceland' 'Norway' 'Switzerland']
Index(['Name', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
       '2023', '2024'],
      dtype='object')


C:\Users\praga\AppData\Local\Temp\ipykernel_33312\3868485910.py:27: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\praga\AppData\Local\Temp\ipykernel_33312\3868485910.py:31: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [47]:
import plotly.express as px

# 1. Ensure the data is sorted chronologically for the slider
df = df.sort_values(by='quarter') # Replace with your actual column name

fig = px.choropleth(
    df, 
    locations='country', 
    locationmode='country names', 
    color='price_index',
    color_continuous_scale=['green', 'yellow', 'red'],
    hover_data={'price_index': ':.2f', 'quarter': True},
    animation_frame='quarter', # The slider will use your "2022-Q1" column
    range_color=[df['price_index'].min(), 300], # Keep scale consistent
    title='European Housing Price Index Over Time'
)

fig.update_layout(
    autosize=True,
    width=800,
    height=700, # Increased height slightly to accommodate the slider/play button
    margin={"r":0, "t":50, "l":0, "b":0},
    geo=dict(
        scope='europe',
        projection_type='mercator',
        lataxis_range=[34, 70], 
        lonaxis_range=[-20, 40],
        landcolor="#f3f5f2",
        showland=True,
        showocean=True,
        oceancolor="#e8f4f8",
        showcountries=True,
        showlakes=True,
        lakecolor="#e8f4f8",
        showframe=False,
        resolution=50,
    ),
    
)

fig.show()
fig.write_html('european_housing_prices_choropleth.html')

C:\Users\praga\AppData\Local\Temp\ipykernel_33312\657663353.py:6: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



In [48]:
# Ensure years are strings to match column names if they were imported as text
years_all = [str(y) for y in range(2015, 2025)]
years_to_plot = ['2022', '2023', '2024']

groups = {
    'Single': df_snk,
    'Double No Kids': df_dnk,
    'Double With Kids': df_dwk
}

combined_list = []

for label, df_2 in groups.items():
    # 1. Force years to numeric (crucial for math and plotting)
    df_2[years_all] = df_2[years_all].apply(pd.to_numeric, errors='coerce')
    
    # 2. Re-apply indexing if not already done (Base 2015 = 100)
    # This ensures we are definitely using the "Index" and not "Income"
    df_2[years_all] = df_2[years_all].div(df_2['2015'], axis=0) * 100
    
    # 3. Melt for Plotly (Long Format)
    melted = df_2.melt(id_vars=['Name'], value_vars=years_to_plot, 
                       var_name='Year', value_name='Income_Index')
    melted['Income Group'] = label
    combined_list.append(melted)

df_plot = pd.concat(combined_list).dropna(subset=['Income_Index'])

C:\Users\praga\AppData\Local\Temp\ipykernel_33312\3319769758.py:15: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\praga\AppData\Local\Temp\ipykernel_33312\3319769758.py:19: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [49]:
# Calculate consistent color bounds for the Index
min_idx = df_plot['Income_Index'].min()
max_idx = df_plot['Income_Index'].max()

fig = px.choropleth(
    df_plot, 
    locations='Name', 
    locationmode='country names', 
    color='Income_Index',
    facet_col='Income Group',
    animation_frame='Year', 
    color_continuous_scale=['green', 'yellow', 'red'],
    range_color=[100, 300],
    title='Net Income Growth Index (Base Year 2015 = 100)',
    labels={'Income_Index': 'Growth Index'},
    hover_name='Name',
    hover_data={'Income_Index': ':.2f', 'Year': True}
)

fig.update_layout(
    width=1400, 
    height=600,
    margin={"r":10, "t":80, "l":10, "b":10}
)

fig.update_geos(
    projection_type="mercator",
    scope='europe',
    lataxis_range=[34, 70], 
    lonaxis_range=[-20, 40]
)

fig.show()
fig.write_html('european_income_index_choropleth.html')

C:\Users\praga\AppData\Local\Temp\ipykernel_33312\2877862509.py:5: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



In [52]:
print(df.columns)
df_h_2024 = df[df['quarter'] == '2024-Q4'].groupby('country')['price_index'].mean().reset_index()

# 2. Match and Calculate
income_groups = {'Single': df_snk, 'Double No Kids': df_dnk, 'Double With Kids': df_dwk}
affordability_list = []

for label, df_inc in income_groups.items():
    temp = df_inc[['Name', '2024']].copy()
    temp.columns = ['country', 'income_index_2024']
    merged = temp.merge(df_h_2024, on='country')
    
    # NEW CALCULATION: Percentage change relative to 2015 baseline
    # Result of 10% means "10% easier than 2015"
    # Result of -15% means "15% harder than 2015"
    merged['Affordability_Pct_Change'] = ((merged['income_index_2024'] / merged['price_index']) - 1) * 100
    merged['Group'] = label
    
    # Create a color category for easy discrete coloring
    merged['Status'] = merged['Affordability_Pct_Change'].apply(lambda x: 'Easier' if x >= 0 else 'Harder')
    
    affordability_list.append(merged)

df_plot = pd.concat(affordability_list)

# 2. Create the Diverging Bar Chart
fig = px.bar(
    df_plot, 
    x='country', 
    y='Affordability_Pct_Change', 
    color='Status', # This splits the colors based on the calculation
    facet_row='Group', # Separates the three income groups for clarity
    color_discrete_map={'Easier': '#2ca02c', 'Harder': '#d62728'}, # Explicit Green and Red
    title='Change in Housing Affordability (2024 vs. 2015 Baseline)',
    labels={'Affordability_Pct_Change': 'Affordability Change (%)', 'country': 'Country'},
    category_orders={"Status": ["Easier", "Harder"]}
)

# 3. Styling for the "Diverging" look
fig.update_layout(
    height=900,
    showlegend=True,
    yaxis_title="Percentage Change (%)",
    # Set the y-axis to be centered or at least show the 0 line clearly
    yaxis=dict(zeroline=True, zerolinewidth=2, zerolinecolor='Black')
)

# Clean up axis labels across all facets
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(tickangle=-45)

fig.show()
fig.write_html('european_affordability_change_bar_chart.html')

Index(['country', 'country_type', 'eu_member', 'eurozone_member', 'year',
       'quarter_num', 'quarter', 'price_index', 'quarterly_change_pct',
       'yearly_change_pct', 'price_change_since_2015_pct', 'data_quality',
       'dt'],
      dtype='object')


In [ ]:
import folium
import requests
import pandas as pd
from branca.element import IFrame

# =========================
# 1. SETUP DATA
# =========================

latest_quarter = "2025-Q3"
df_latest = df[df['quarter'] == latest_quarter].copy()
df['dt'] = pd.to_datetime(df['dt'], errors='coerce')

# =========================
# 2. LOAD GEOJSON
# =========================

geojson_url = "https://raw.githubusercontent.com/leakyMirror/map-of-europe/master/GeoJSON/europe.geojson"
geojson_data = requests.get(geojson_url).json()

# =========================
# 3. CREATE MAP
# =========================

m = folium.Map(
location=[54, 15],
    zoom_start=4,
    tiles='cartodbpositron',
    
    zoom_control=False,
    scrollWheelZoom=False,
    dragging=False,
   
    max_bounds=True,
    min_lat=33,
    max_lat=72,
    min_lon=-25,
    max_lon=45,
    min_zoom=4
)

m.fit_bounds([[34.0, -25.0], [72.0, 45.0]])

# =========================
# 4. CHOROPLETH (BASE)
# =========================

folium.Choropleth(
    geo_data=geojson_data,
    data=df_latest,
    columns=["country", "price_index"],
    key_on="feature.properties.NAME",
    fill_color="RdYlGn_r",
    fill_opacity=0.7,
    line_opacity=0.2,
    nan_fill_opacity=0.0
).add_to(m)

# =========================
# 5. ADD INTERACTIVE POPUPS (FIXED)
# =========================

for feature in geojson_data['features']:
    country_name = feature['properties']['NAME']
    history = df[df['country'] == country_name].sort_values('dt')

    if history.empty:
        continue

    # Create plotly chart
    fig = px.line(
        history,
        x='dt',
        y='price_index',
        title=country_name,
        markers=True,
        template='plotly_white'
    )

    fig.update_layout(
        width=350,
        height=250,
        margin=dict(l=10, r=10, t=30, b=10)
    )

    html = fig.to_html(include_plotlyjs='cdn', full_html=False)

    iframe = IFrame(html=html, width=370, height=270)
    popup = folium.Popup(iframe, max_width=400)

    # Attach popup to that specific country
    folium.GeoJson(
        feature,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': 'transparent',
            'weight': 0
        },
        tooltip=country_name,
        popup=popup
    ).add_to(m)

# =========================
# 6. DISPLAY
# =========================

m
m.save('european_housing_prices_folium.html')